In [1]:
# ============================================================
# УЛУЧШЕННЫЙ CLTV NOTEBOOK - ПОЛНАЯ ВЕРСИЯ
# Часть 1: Импорты, конфигурация, загрузка данных
# ============================================================

# %% ИМПОРТЫ И НАСТРОЙКИ
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os
import warnings
from collections import defaultdict, deque
from scipy import stats
import logging
import pickle
import json
from pathlib import Path

from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import optuna
from optuna.samplers import TPESampler

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")

print("Импорты загружены")

# %% КОНФИГУРАЦИЯ
class Config:
    TRAIN_PATH = "CLTV_UL_TRAIN_MART.csv"
    PROD_PATH = "CLTV_UL_PROD_SNAPSHOT.csv"
    CHURN_PATH = "../churn_result_msb.csv"
    MODEL_DIR = Path("models")
    MODEL_VERSION = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    FORECAST_START = "2025-10-31"
    HORIZON_MONTHS = 6
    DISCOUNT_RATE_ANNUAL = 0.12
    VALIDATION_CUTOFF = "2025-03-31"
    MIN_SAMPLES_PER_SEGMENT = 1000
    
    CATEGORICAL_FEATURES = ['QUALITY_CODE', 'SUBJECT_KIND_ID', 'EC_SECTOR_ID']
    BASE_FEATURES = [
        'MARGIN', 'MARGIN_LAG1', 'MARGIN_LAG2', 'MARGIN_LAG3',
        'MARGIN_AVG_1M_LAG', 'MARGIN_AVG_2M_LAG', 'MARGIN_AVG_3M_LAG',
        'MARGIN_AVG_6M_LAG', 'MARGIN_AVG_12M_LAG', 'MARGIN_STDDEV_12M_LAG',
        'MARGIN_GROWTH_RATE_3M', 'MONTH_OF_YEAR', 'QUARTER_OF_YEAR', 'TENURE_MONTHS'
    ]
    
    OPTUNA_TRIALS = 10
    OPTUNA_TIMEOUT = 1800
    CV_SPLITS = 3
    TRAIN_QUANTILES = False 
    
    @classmethod
    def ensure_directories(cls):
        cls.MODEL_DIR.mkdir(parents=True, exist_ok=True)
        (cls.MODEL_DIR / cls.MODEL_VERSION).mkdir(parents=True, exist_ok=True)

Config.ensure_directories()
print(f"Версия: {Config.MODEL_VERSION}")

# %% ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
def add_months(dt_str, k):
    d = datetime.strptime(dt_str, "%Y-%m-%d")
    return (d + relativedelta(months=+k)).strftime("%Y-%m-%d")

def discounted(value, months, annual_rate):
    if annual_rate <= 0:
        return value
    monthly_rate = (1 + annual_rate) ** (1/12) - 1
    return value / ((1 + monthly_rate) ** months)

def read_table(path):
    if not os.path.exists(path):
        logger.warning(f"Файл не найден: {path}")
        return pd.DataFrame()
    ext = os.path.splitext(path)[1].lower()
    if ext in [".parquet", ".pq", ".parq"]:
        return pd.read_parquet(path)
    elif ext in [".csv", ".txt"]:
        try:
            return pd.read_csv(path, sep='|', encoding="windows-1251", thousands=',')
        except:
            try:
                return pd.read_csv(path, sep=',')
            except:
                return pd.read_csv(path, sep='\t')
    raise ValueError(f"Неподдерживаемый формат: {ext}")

def fix_categorical_features(df, cat_features):
    df_fixed = df.copy()
    for col in cat_features:
        if col in df_fixed.columns:
            df_fixed[col] = df_fixed[col].fillna('UNKNOWN').astype(str)
            df_fixed[col] = df_fixed[col].str.replace('.0', '', regex=False)
    return df_fixed

def stabilize_target(y):
    return np.sign(y) * np.log1p(np.abs(y))

def inverse_stabilize_target(y_stable):
    return np.sign(y_stable) * (np.exp(np.abs(y_stable)) - 1)

def calculate_business_metrics(y_true, y_pred, percentiles=[0.5, 0.75, 0.9, 0.95]):
    metrics = {}
    for p in percentiles:
        threshold = np.quantile(y_true, p)
        mask = y_true >= threshold
        if mask.sum() > 0:
            mae = mean_absolute_error(y_true[mask], y_pred[mask])
            r2 = r2_score(y_true[mask], y_pred[mask])
            metrics[f'mae_top_{int((1-p)*100)}pct'] = mae
            metrics[f'r2_top_{int((1-p)*100)}pct'] = r2
    return metrics

print("Функции загружены")

# %% ЗАГРУЗКА ДАННЫХ
logger.info("Загрузка данных...")
train = read_table(Config.TRAIN_PATH)
prod = read_table(Config.PROD_PATH)
churn_raw = read_table(Config.CHURN_PATH)

print(f"Обучающая: {len(train):,} записей, {train['CLIENT_ID'].nunique():,} клиентов")
print(f"Продакшн: {len(prod):,} записей")
print(f"Churn: {len(churn_raw):,} записей")

# %% ОБРАБОТКА CHURN
def calculate_monthly_hazard_rate(churn_3m_prob):
    churn_3m = float(max(0.0, min(0.99999, churn_3m_prob)))
    survival_3m = 1.0 - churn_3m
    if survival_3m <= 0:
        monthly_hazard = 0.5
    else:
        monthly_hazard = -np.log(survival_3m) / 3.0
    return {
        'monthly_hazard': monthly_hazard,
        'monthly_churn_prob': 1.0 - np.exp(-monthly_hazard),
        'implied_3m_survival': np.exp(-3 * monthly_hazard),
        'original_3m_churn': churn_3m
    }

churn = churn_raw.copy()
if 'churn_probability' in churn.columns:
    churn = churn.rename(columns={'churn_probability': 'CHURN_PROB_3M'})
churn.columns = churn.columns.str.upper()

churn_map = {}
for _, row in churn.iterrows():
    churn_map[row['CLIENT_ID']] = calculate_monthly_hazard_rate(row['CHURN_PROB_3M'])

print(f"Churn map: {len(churn_map):,} клиентов")

def calculate_survival_probability(client_id, churn_map):
    if client_id not in churn_map:
        return 1.0
    return np.exp(-churn_map[client_id]['monthly_hazard'])

# %% ПРЕДОБРАБОТКА
all_categorical = Config.CATEGORICAL_FEATURES + ['SEGMENT_ID']
train_fixed = fix_categorical_features(train, all_categorical)
prod_fixed = fix_categorical_features(prod, all_categorical)

ALL_FEATURES = ['SEGMENT_ID'] + Config.BASE_FEATURES + Config.CATEGORICAL_FEATURES
available_features = [f for f in ALL_FEATURES if f in train_fixed.columns]

numeric_features = [f for f in available_features if f not in all_categorical]
for col in numeric_features:
    train_fixed[col] = train_fixed[col].fillna(0.0)
    if col in prod_fixed.columns:
        prod_fixed[col] = prod_fixed[col].fillna(0.0)

print(f"Фичи готовы: {len(available_features)}")
print(f"Сегменты: {sorted(train_fixed['SEGMENT_ID'].unique())}")

2025-11-09 23:37:43,090 - INFO - Загрузка данных...


Импорты загружены
Версия: 20251109_233743
Функции загружены
Обучающая: 4,052,063 записей, 233,474 клиентов
Продакшн: 358,787 записей
Churn: 163,592 записей
Churn map: 163,592 клиентов
Фичи готовы: 18
Сегменты: ['1022', '1023', '1026', '1027', '1028', '1040']


In [2]:
# ============================================================
# ЧАСТЬ 2: КЛАСС УЛУЧШЕННОГО СЕГМЕНТИРОВАННОГО CLTV
# Вставить после Части 1
# ============================================================

# %% КЛАСС УЛУЧШЕННОГО CLTV
class ImprovedSegmentedCLTV:
    """Улучшенная версия с Optuna, квантилями и сохранением"""
    
    def __init__(self, min_samples_per_segment=1000, use_optuna=True, 
                 n_trials=30, cv_splits=3):
        self.models = {}
        self.quantile_models = {}
        self.segment_stats = {}
        self.min_samples_per_segment = min_samples_per_segment
        self.fallback_segments = {}
        self.feature_importance = {}
        self.use_optuna = use_optuna
        self.n_trials = n_trials
        self.cv_splits = cv_splits
        self.best_params = {}
        self.metadata = {}
        
    def analyze_segment_distribution(self, df, segment_col='SEGMENT_ID'):
        segment_stats = df.groupby(segment_col).agg({
            'CLIENT_ID': 'nunique',
            'TARGET_NEXT_MARGIN': ['count', 'mean', 'std', 'min', 'max']
        }).round(2)
        
        segment_stats.columns = ['unique_clients', 'total_records', 'avg_margin', 
                               'std_margin', 'min_margin', 'max_margin']
        segment_stats['records_per_client'] = (
            segment_stats['total_records'] / segment_stats['unique_clients']
        ).round(1)
        
        small = segment_stats[segment_stats['total_records'] < self.min_samples_per_segment].index.tolist()
        large = segment_stats[segment_stats['total_records'] >= self.min_samples_per_segment].index.tolist()
        
        print("Анализ сегментов:")
        print(segment_stats)
        print(f"\nБольшие сегменты (>={self.min_samples_per_segment}): {large}")
        print(f"Малые сегменты (<{self.min_samples_per_segment}): {small}")
        
        return segment_stats, large, small
    
    def prepare_segment_data(self, df, segment_id, features, target_col, validation_cutoff):
        segment_data = df[df['SEGMENT_ID'] == segment_id].copy()
        segment_data['target_stable'] = stabilize_target(segment_data[target_col])
        
        train_mask = pd.to_datetime(segment_data['MONTH_END']) <= pd.to_datetime(validation_cutoff)
        val_mask = ~train_mask
        
        features_for_segment = [f for f in features if f != 'SEGMENT_ID']
        
        X_train = segment_data[train_mask][features_for_segment]
        y_train = segment_data[train_mask]['target_stable']
        X_val = segment_data[val_mask][features_for_segment] 
        y_val = segment_data[val_mask]['target_stable']
        y_val_original = segment_data[val_mask][target_col]
        
        return X_train, y_train, X_val, y_val, y_val_original, segment_data
    
    def optimize_hyperparameters_optuna(self, segment_id, X_train, y_train, X_val, y_val, cat_features):
        cat_indices = []
        features_list = X_train.columns.tolist()
        for cat_feat in cat_features:
            if cat_feat in features_list:
                cat_indices.append(features_list.index(cat_feat))
        
        def objective(trial):
            params = {
                'iterations': trial.suggest_int('iterations', 500, 3000),
                'depth': trial.suggest_int('depth', 4, 8),
                'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
                'l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 100),
                'random_seed': 42,
                'loss_function': 'MAE',
                'verbose': False,
                'early_stopping_rounds': 50
            }
            
            train_pool = Pool(X_train, y_train, cat_features=cat_indices)
            val_pool = Pool(X_val, y_val, cat_features=cat_indices)
            
            model = CatBoostRegressor(**params)
            model.fit(train_pool, eval_set=val_pool, use_best_model=True)
            
            return mean_absolute_error(y_val, model.predict(X_val))
        
        study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
        study.optimize(objective, n_trials=self.n_trials, timeout=Config.OPTUNA_TIMEOUT, 
                      show_progress_bar=True)
        
        print(f"  Лучший MAE: {study.best_value:.3f}")
        return study.best_params
    
    def get_default_params(self, segment_id, data_size):
        base = {"random_seed": 42, "loss_function": "MAE", "verbose": False, 
                "early_stopping_rounds": 50}
        if data_size > 100000:
            base.update({"iterations": 2000, "depth": 6, "learning_rate": 0.01, "l2_leaf_reg": 20})
        elif data_size > 10000:
            base.update({"iterations": 1500, "depth": 5, "learning_rate": 0.015, "l2_leaf_reg": 30})
        else:
            base.update({"iterations": 1000, "depth": 4, "learning_rate": 0.02, "l2_leaf_reg": 50})
        return base
    
    def train_segment_model(self, segment_id, X_train, y_train, X_val, y_val, categorical_features=None):
        cat_indices = []
        if categorical_features:
            features_list = X_train.columns.tolist()
            for cat_feat in categorical_features:
                if cat_feat in features_list:
                    cat_indices.append(features_list.index(cat_feat))
        
        if self.use_optuna:
            print(f"  Optuna ({self.n_trials} trials)...")
            best_params = self.optimize_hyperparameters_optuna(
                segment_id, X_train, y_train, X_val, y_val, categorical_features
            )
            best_params.update({'random_seed': 42, 'loss_function': 'MAE', 'verbose': False})
        else:
            best_params = self.get_default_params(segment_id, len(X_train))
        
        self.best_params[segment_id] = best_params
        
        train_pool = Pool(X_train, y_train, cat_features=cat_indices)
        val_pool = Pool(X_val, y_val, cat_features=cat_indices)
        
        model = CatBoostRegressor(**best_params)
        model.fit(train_pool, eval_set=val_pool, use_best_model=True)
        
        # Квантильные модели
        quantile_models = {}
        if Config.TRAIN_QUANTILES:
            print(f"  Обучение квантилей (0.1, 0.5, 0.9)...")
            for quantile in [0.1, 0.5, 0.9]:
                q_params = best_params.copy()
                q_params['loss_function'] = f'Quantile:alpha={quantile}'
                q_model = CatBoostRegressor(**q_params)
                q_model.fit(train_pool, eval_set=val_pool, use_best_model=True)
                quantile_models[quantile] = q_model
        else:
            print(f"  Квантили отключены")
        
        train_pred = model.predict(X_train)
        val_pred = model.predict(X_val)
        
        metrics = {
            'train_r2': r2_score(y_train, train_pred),
            'val_r2': r2_score(y_val, val_pred),
            'val_mae': mean_absolute_error(y_val, val_pred),
            'train_samples': len(X_train),
            'val_samples': len(X_val),
            'feature_importance': pd.DataFrame({
                'feature': X_train.columns,
                'importance': model.feature_importances_
            }).sort_values('importance', ascending=False),
            'business_metrics': calculate_business_metrics(y_val, val_pred),
            'best_params': best_params
        }
        
        return model, quantile_models, metrics
    
    def predict_segment(self, segment_id, X, return_stable=False, return_uncertainty=False):
        if segment_id in self.models:
            model = self.models[segment_id]
            X_pred = X.drop('SEGMENT_ID', axis=1) if 'SEGMENT_ID' in X.columns and segment_id != 'FALLBACK' else X
        elif segment_id in self.fallback_segments:
            model = self.models[self.fallback_segments[segment_id]]
            X_pred = X
        else:
            raise ValueError(f"Нет модели для {segment_id}")
        
        pred_stable = model.predict(X_pred)
        result = {'prediction': pred_stable}
        
        if return_uncertainty:
            if segment_id in self.quantile_models and len(self.quantile_models[segment_id]) > 0:
                q_models = self.quantile_models[segment_id]
                result['lower_bound'] = q_models[0.1].predict(X_pred)
                result['median'] = q_models[0.5].predict(X_pred)
                result['upper_bound'] = q_models[0.9].predict(X_pred)
            else:
                # Если квантилей нет - используем простую эвристику
                # ±30% от прогноза как грубая оценка неопределенности
                result['lower_bound'] = pred_stable * 0.7
                result['median'] = pred_stable
                result['upper_bound'] = pred_stable * 1.3
        
        if not return_stable:
            result['prediction'] = inverse_stabilize_target(result['prediction'])
            if 'lower_bound' in result:
                result['lower_bound'] = inverse_stabilize_target(result['lower_bound'])
                result['median'] = inverse_stabilize_target(result['median'])
                result['upper_bound'] = inverse_stabilize_target(result['upper_bound'])
        
        return result['prediction'] if not return_uncertainty else result
    
    def save_models(self, path, version, features, churn_map):
        save_path = Path(path) / version
        save_path.mkdir(parents=True, exist_ok=True)
        
        metadata = {
            'version': version,
            'train_date': datetime.now().isoformat(),
            'segments': list(self.models.keys()),
            'features': features,
            'fallback_segments': self.fallback_segments,
            'best_params': self.best_params
        }
        
        metrics_data = {}
        for seg_id, stats in self.segment_stats.items():
            metrics_data[seg_id] = {
                'train_r2': float(stats['train_r2']),
                'val_r2': float(stats['val_r2']),
                'val_mae': float(stats['val_mae']),
                'train_samples': int(stats['train_samples']),
                'val_samples': int(stats['val_samples']),
                'business_metrics': stats['business_metrics']
            }
        metadata['segment_metrics'] = metrics_data
        
        for seg_id, model in self.models.items():
            model.save_model(str(save_path / f"model_seg_{seg_id}.cbm"))
        
        for seg_id, q_models in self.quantile_models.items():
            for q, q_model in q_models.items():
                q_model.save_model(str(save_path / f"model_seg_{seg_id}_q{q}.cbm"))
        
        with open(save_path / "metadata.json", 'w') as f:
            json.dump(metadata, f, indent=2)
        
        for seg_id, importance_df in self.feature_importance.items():
            importance_df.to_csv(save_path / f"feature_importance_{seg_id}.csv", index=False)
        
        with open(save_path / "churn_map.pkl", 'wb') as f:
            pickle.dump(churn_map, f)
        
        with open(save_path / "model_object.pkl", 'wb') as f:
            pickle.dump(self, f)
        
        logger.info(f"Модели сохранены: {save_path}")
        return save_path
    
    @classmethod
    def load_models(cls, path, version):
        load_path = Path(path) / version
        if not load_path.exists():
            raise ValueError(f"Не найдено: {load_path}")
        
        with open(load_path / "model_object.pkl", 'rb') as f:
            model_obj = pickle.load(f)
        
        for seg_id in model_obj.models.keys():
            model_path = load_path / f"model_seg_{seg_id}.cbm"
            if model_path.exists():
                model_obj.models[seg_id] = CatBoostRegressor()
                model_obj.models[seg_id].load_model(str(model_path))
        
        logger.info(f"Модели загружены: {load_path}")
        return model_obj

print("Класс ImprovedSegmentedCLTV создан")

Класс ImprovedSegmentedCLTV создан


In [3]:
# ============================================================
# ЧАСТЬ 3: ОБУЧЕНИЕ МОДЕЛЕЙ
# Вставить после Части 2
# ============================================================

# %% ФУНКЦИЯ ОБУЧЕНИЯ ВСЕХ СЕГМЕНТОВ
def fit_all_segments_improved(segmented_model, df, features, target_col='TARGET_NEXT_MARGIN', 
                              validation_cutoff='2025-03-31', categorical_features=None):
    """Обучение моделей для всех сегментов"""
    
    segment_stats, large_segments, small_segments = segmented_model.analyze_segment_distribution(df)
    
    print(f"\nНачинаем обучение...")
    results = {}
    
    # Большие сегменты
    for segment_id in large_segments:
        print(f"\n{'='*60}")
        print(f"Сегмент {segment_id}")
        print(f"{'='*60}")
        
        try:
            X_train, y_train, X_val, y_val, y_val_orig, segment_data = \
                segmented_model.prepare_segment_data(df, segment_id, features, target_col, validation_cutoff)
            
            if len(X_train) < 100:
                print(f"  Недостаточно данных: {len(X_train)}")
                continue
            
            model, quantile_models, metrics = segmented_model.train_segment_model(
                segment_id, X_train, y_train, X_val, y_val, categorical_features
            )
            
            segmented_model.models[segment_id] = model
            segmented_model.quantile_models[segment_id] = quantile_models
            segmented_model.segment_stats[segment_id] = metrics
            segmented_model.feature_importance[segment_id] = metrics['feature_importance']
            
            print(f"\n  Результаты:")
            print(f"    R² train: {metrics['train_r2']:.3f}, R² val: {metrics['val_r2']:.3f}")
            print(f"    MAE val: {metrics['val_mae']:.3f}")
            print(f"    Samples: {metrics['train_samples']:,} / {metrics['val_samples']:,}")
            
            print(f"    Бизнес метрики:")
            for metric_name, value in metrics['business_metrics'].items():
                print(f"      {metric_name}: {value:.3f}")
            
            results[segment_id] = metrics
            
        except Exception as e:
            print(f"  Ошибка: {e}")
            continue
    
    # Fallback для малых сегментов
    if small_segments:
        print(f"\n{'='*60}")
        print(f"Fallback для малых сегментов: {small_segments}")
        print(f"{'='*60}")
        
        small_segments_data = df[df['SEGMENT_ID'].isin(small_segments)].copy()
        
        if len(small_segments_data) >= 500:
            try:
                small_segments_data['target_stable'] = stabilize_target(small_segments_data[target_col])
                
                train_mask = pd.to_datetime(small_segments_data['MONTH_END']) <= pd.to_datetime(validation_cutoff)
                val_mask = ~train_mask
                
                X_train_fb = small_segments_data[train_mask][features]
                y_train_fb = small_segments_data[train_mask]['target_stable']
                X_val_fb = small_segments_data[val_mask][features] 
                y_val_fb = small_segments_data[val_mask]['target_stable']
                
                model_fb, quantile_models_fb, metrics_fb = segmented_model.train_segment_model(
                    'FALLBACK', X_train_fb, y_train_fb, X_val_fb, y_val_fb, categorical_features
                )
                
                segmented_model.models['FALLBACK'] = model_fb
                segmented_model.quantile_models['FALLBACK'] = quantile_models_fb
                segmented_model.segment_stats['FALLBACK'] = metrics_fb
                
                for seg_id in small_segments:
                    segmented_model.fallback_segments[seg_id] = 'FALLBACK'
                
                print(f"  Fallback R² val: {metrics_fb['val_r2']:.3f}")
                
            except Exception as e:
                print(f"  Ошибка fallback: {e}")
    
    return results

# %% ЗАПУСК ОБУЧЕНИЯ
logger.info("Запуск обучения...")

improved_cltv = ImprovedSegmentedCLTV(
    min_samples_per_segment=Config.MIN_SAMPLES_PER_SEGMENT,
    use_optuna=True,  # Установить False для быстрого тестирования
    n_trials=Config.OPTUNA_TRIALS,
    cv_splits=Config.CV_SPLITS
)

segment_results = fit_all_segments_improved(
    improved_cltv,
    df=train_fixed,
    features=available_features,
    target_col='TARGET_NEXT_MARGIN',
    validation_cutoff=Config.VALIDATION_CUTOFF,
    categorical_features=Config.CATEGORICAL_FEATURES
)

# %% СВОДКА РЕЗУЛЬТАТОВ
print("\n" + "="*70)
print("СВОДКА ПО МОДЕЛЯМ")
print("="*70)

summary_data = []
for segment_id, stats in improved_cltv.segment_stats.items():
    summary_data.append({
        'Segment': segment_id,
        'Train_R2': f"{stats['train_r2']:.3f}",
        'Val_R2': f"{stats['val_r2']:.3f}",
        'Val_MAE': f"{stats['val_mae']:.1f}",
        'Samples': f"{stats['train_samples']:,} / {stats['val_samples']:,}"
    })

if summary_data:
    print(pd.DataFrame(summary_data).to_string(index=False))
    
    avg_r2 = np.mean([stats['val_r2'] for stats in improved_cltv.segment_stats.values()])
    print(f"\nСредний R²: {avg_r2:.3f}")
    
    baseline_r2 = 0.40
    improvement = (avg_r2 - baseline_r2) / baseline_r2 * 100
    print(f"Улучшение vs baseline ({baseline_r2:.2f}): {improvement:+.1f}%")
else:
    print("Нет обученных моделей!")

# %% СОХРАНЕНИЕ МОДЕЛЕЙ
if len(improved_cltv.models) > 0:
    logger.info("Сохранение моделей...")
    
    save_path = improved_cltv.save_models(
        path=Config.MODEL_DIR,
        version=Config.MODEL_VERSION,
        features=available_features,
        churn_map=churn_map
    )
    
    print(f"\nМодели сохранены:")
    print(f"  Путь: {save_path}")
    print(f"  Версия: {Config.MODEL_VERSION}")
    print(f"  Моделей: {len(improved_cltv.models)}")
else:
    print("Нет моделей для сохранения")

2025-11-09 23:38:09,706 - INFO - Запуск обучения...


Анализ сегментов:
            unique_clients  total_records   avg_margin    std_margin  \
SEGMENT_ID                                                             
1022                 11942         200974   577,062.74  2,464,777.62   
1023                  1340          21424 3,320,837.00 31,419,946.39   
1026                 83416         955279    24,270.71    343,743.75   
1027                212068        2874328    40,986.39    463,762.62   
1028                     1              8     4,338.05      2,039.63   
1040                     3             50   929,922.18  1,244,860.29   

                min_margin     max_margin  records_per_client  
SEGMENT_ID                                                     
1022       -308,958,453.67 235,343,062.49               16.80  
1023       -850,876,876.86 541,208,955.36               16.00  
1026       -137,561,728.87  83,546,366.51               11.50  
1027       -107,042,025.61 153,569,236.22               13.60  
1028              2,9

[I 2025-11-09 23:38:10,796] A new study created in memory with name: no-name-bab578af-7528-4af9-8944-5881bc9eadd0


  Optuna (10 trials)...


  0%|          | 0/10 [00:00<?, ?it/s]

...

[I 2025-11-10 01:09:12,416] Trial 0 finished with value: 0.5107615198104515 and parameters: {'iterations': 1436, 'depth': 8, 'learning_rate': 0.029106359131330698, 'l2_leaf_reg': 60}. Best is trial 0 with value: 0.5107615198104515.
[I 2025-11-10 01:18:31,947] Trial 1 finished with value: 1.5939530881495145 and parameters: {'iterations': 890, 'depth': 4, 'learning_rate': 0.0013066739238053278, 'l2_leaf_reg': 87}. Best is trial 0 with value: 0.5107615198104515.
[I 2025-11-10 01:49:23,630] Trial 2 finished with value: 0.9619913392080577 and parameters: {'iterations': 2003, 'depth': 7, 'learning_rate': 0.0010994335574766201, 'l2_leaf_reg': 97}. Best is trial 0 with value: 0.5107615198104515.
  Лучший MAE: 0.511
  Квантили отключены


2025-11-10 02:13:56,363 - INFO - Сохранение моделей...



  Результаты:
    R² train: 0.844, R² val: 0.879
    MAE val: 0.509
    Samples: 2,334,023 / 540,305
    Бизнес метрики:
      mae_top_50pct: 0.600
      r2_top_50pct: 0.305
      mae_top_25pct: 0.518
      r2_top_25pct: -0.922
      mae_top_9pct: 0.543
      r2_top_9pct: -3.189
      mae_top_5pct: 0.593
      r2_top_5pct: -5.627

Fallback для малых сегментов: ['1028', '1040']

СВОДКА ПО МОДЕЛЯМ
Segment Train_R2 Val_R2 Val_MAE             Samples
   1022    0.794  0.772     0.7    146,431 / 54,543
   1023    0.674  0.633     1.1      16,349 / 5,075
   1026    0.743  0.778     0.7   584,866 / 370,413
   1027    0.844  0.879     0.5 2,334,023 / 540,305

Средний R²: 0.766
Улучшение vs baseline (0.40): +91.4%


2025-11-10 02:13:58,316 - INFO - Модели сохранены: models\20251109_233743



Модели сохранены:
  Путь: models\20251109_233743
  Версия: 20251109_233743
  Моделей: 4


In [4]:
# ============================================================
# ЧАСТЬ 4: ПРОГНОЗИРОВАНИЕ И АНАЛИЗ РЕЗУЛЬТАТОВ
# Вставить после Части 3
# ============================================================

# %% ФУНКЦИИ ДЛЯ ПРОГНОЗИРОВАНИЯ
def calculate_rolling_features(history, feature_name):
    hist_array = np.array(list(history))
    if len(hist_array) == 0:
        return 0.0
    
    if feature_name.startswith('avg_'):
        window_map = {'1m': 1, '2m': 2, '3m': 3, '6m': 6, '12m': 12}
        window = min(window_map.get(feature_name.split('_')[-1], len(hist_array)), len(hist_array))
        return np.mean(hist_array[-window:]) if window > 0 else 0.0
    
    elif feature_name == 'stddev_12m':
        window = min(12, len(hist_array))
        return np.std(hist_array[-window:]) if window > 1 else 0.0
    
    elif feature_name == 'growth_rate_3m':
        if len(hist_array) >= 6:
            recent_avg = np.mean(hist_array[-2:])
            older_avg = np.mean(hist_array[-6:-3])
            if abs(older_avg) > 1e-6:
                return (recent_avg - older_avg) / abs(older_avg)
        return 0.0
    
    return 0.0

def update_client_state(state, client_id, predicted_margin, month_index):
    client_state = state[client_id]
    client_state['lag3'] = client_state['lag2']
    client_state['lag2'] = client_state['lag1'] 
    client_state['lag1'] = predicted_margin
    client_state['history'].append(predicted_margin)

def initialize_client_state(prod_data):
    state = {}
    for _, row in prod_data.iterrows():
        client_id = int(row['CLIENT_ID'])
        hist = deque(maxlen=12)
        for lag_col in ['MARGIN_LAG3', 'MARGIN_LAG2', 'MARGIN_LAG1', 'MARGIN']:
            if lag_col in row.index and pd.notna(row[lag_col]):
                hist.append(float(row[lag_col]))
        
        if len(hist) == 0:
            hist.append(float(row['MARGIN']))
        
        state[client_id] = {
            'segment_id': str(row['SEGMENT_ID']),
            'forecast_start': pd.to_datetime(Config.FORECAST_START),
            'lag1': row['MARGIN'],
            'lag2': row.get('MARGIN_LAG1', 0.0),
            'lag3': row.get('MARGIN_LAG2', 0.0),
            'history': hist,
            'cumulative_survival': 1.0,
            'base_tenure': row.get('TENURE_MONTHS', 1),
            'quality_code': str(row.get('QUALITY_CODE', 'UNKNOWN')),
            'subject_kind_id': str(row.get('SUBJECT_KIND_ID', 'UNKNOWN')),
            'ec_sector_id': str(row.get('EC_SECTOR_ID', 'UNKNOWN')),
        }
    return state

# %% РЕКУРСИВНОЕ ПРОГНОЗИРОВАНИЕ
def improved_recursive_forecasting(segmented_model_obj, prod_data, churn_map, 
                                  horizon_months=6, forecast_start="2025-09-30",
                                  include_uncertainty=True):
    """Рекурсивное прогнозирование с uncertainty"""
    
    prod_data_fixed = fix_categorical_features(prod_data, all_categorical)
    prod_with_churn = prod_data_fixed[prod_data_fixed['CLIENT_ID'].isin(churn_map.keys())].copy()
    
    print(f"Данные для прогнозирования:")
    print(f"  Всего: {len(prod_data_fixed):,}")
    print(f"  С churn: {len(prod_with_churn):,}")
    
    client_state = initialize_client_state(prod_with_churn)
    start_month_end = pd.to_datetime(forecast_start)
    forecast_records = []
    
    print(f"\nПрогноз на {horizon_months} месяцев...")
    
    for month_idx in range(1, horizon_months + 1):
        if month_idx % 2 == 1:
            print(f"Месяц {month_idx}/{horizon_months}")
        
        current_month_end = start_month_end + relativedelta(months=month_idx-1)
        month_of_year = current_month_end.month
        quarter_of_year = ((current_month_end.month - 1) // 3) + 1
        
        clients_by_segment = defaultdict(list)
        client_features_by_segment = defaultdict(list)
        
        for client_id, state in client_state.items():
            segment_id = state['segment_id']
            
            features = {
                'SEGMENT_ID': str(segment_id),
                'MARGIN': state['lag1'],
                'MARGIN_LAG1': state['lag1'], 
                'MARGIN_LAG2': state['lag2'],
                'MARGIN_LAG3': state['lag3'],
                'MARGIN_AVG_1M_LAG': calculate_rolling_features(state['history'], 'avg_1m'),
                'MARGIN_AVG_2M_LAG': calculate_rolling_features(state['history'], 'avg_2m'),
                'MARGIN_AVG_3M_LAG': calculate_rolling_features(state['history'], 'avg_3m'),
                'MARGIN_AVG_6M_LAG': calculate_rolling_features(state['history'], 'avg_6m'),
                'MARGIN_AVG_12M_LAG': calculate_rolling_features(state['history'], 'avg_12m'),
                'MARGIN_STDDEV_12M_LAG': calculate_rolling_features(state['history'], 'stddev_12m'),
                'MARGIN_GROWTH_RATE_3M': calculate_rolling_features(state['history'], 'growth_rate_3m'),
                'MONTH_OF_YEAR': month_of_year,
                'QUARTER_OF_YEAR': quarter_of_year, 
                'TENURE_MONTHS': state['base_tenure'] + month_idx,
                'QUALITY_CODE': state['quality_code'],
                'SUBJECT_KIND_ID': state['subject_kind_id'],
                'EC_SECTOR_ID': state['ec_sector_id'],
            }
            
            clients_by_segment[segment_id].append(client_id)
            client_features_by_segment[segment_id].append(features)
        
        all_predictions = {}
        all_uncertainty = {}
        
        for segment_id, client_ids in clients_by_segment.items():
            if not client_ids:
                continue
                
            segment_features_df = pd.DataFrame(client_features_by_segment[segment_id])
            
            try:
                if include_uncertainty:
                    pred_result = segmented_model_obj.predict_segment(
                        segment_id, segment_features_df, 
                        return_stable=False, return_uncertainty=True
                    )
                    
                    for i, client_id in enumerate(client_ids):
                        all_predictions[client_id] = pred_result['prediction'][i]
                        all_uncertainty[client_id] = {
                            'lower': pred_result['lower_bound'][i],
                            'median': pred_result['median'][i],
                            'upper': pred_result['upper_bound'][i]
                        }
                else:
                    segment_predictions = segmented_model_obj.predict_segment(
                        segment_id, segment_features_df, return_stable=False
                    )
                    for i, client_id in enumerate(client_ids):
                        all_predictions[client_id] = segment_predictions[i]
                    
            except Exception as e:
                print(f"Ошибка {segment_id}: {e}")
                avg_margin = np.mean([state['lag1'] for state in client_state.values() 
                                    if state['segment_id'] == segment_id])
                for client_id in client_ids:
                    all_predictions[client_id] = avg_margin
        
        for client_id, predicted_margin in all_predictions.items():
            monthly_survival = calculate_survival_probability(client_id, churn_map)
            client_state[client_id]['cumulative_survival'] *= monthly_survival
            
            margin_with_survival = predicted_margin * client_state[client_id]['cumulative_survival']
            margin_discounted = discounted(margin_with_survival, month_idx, Config.DISCOUNT_RATE_ANNUAL)
            
            record = {
                'CLIENT_ID': client_id,
                'SEGMENT_ID': client_state[client_id]['segment_id'],
                'FORECAST_MONTH_END': current_month_end.strftime('%Y-%m-%d'),
                'MONTH_INDEX': month_idx,
                'PRED_MARGIN_RAW': float(predicted_margin),
                'MONTHLY_SURVIVAL': float(monthly_survival),
                'CUM_SURVIVAL': float(client_state[client_id]['cumulative_survival']),
                'PRED_MARGIN_SURV': float(margin_with_survival),
                'PRED_MARGIN_SURV_DISCOUNTED': float(margin_discounted)
            }
            
            if include_uncertainty and client_id in all_uncertainty:
                unc = all_uncertainty[client_id]
                record['PRED_LOWER_BOUND'] = float(unc['lower'] * client_state[client_id]['cumulative_survival'])
                record['PRED_UPPER_BOUND'] = float(unc['upper'] * client_state[client_id]['cumulative_survival'])
            
            forecast_records.append(record)
            update_client_state(client_state, client_id, predicted_margin, month_idx)
    
    print(f"Завершено: {len(forecast_records):,} записей")
    return pd.DataFrame(forecast_records)

# %% ЗАПУСК ПРОГНОЗИРОВАНИЯ
if len(improved_cltv.models) > 0:
    logger.info("Запуск прогнозирования...")
    
    improved_forecasts = improved_recursive_forecasting(
        improved_cltv, 
        prod_fixed, 
        churn_map,
        horizon_months=Config.HORIZON_MONTHS,
        forecast_start=Config.FORECAST_START,
        include_uncertainty=Config.TRAIN_QUANTILES
    )
    
    print(f"\nРезультаты:")
    print(f"  Прогнозов: {len(improved_forecasts):,}")
    print(f"  Клиентов: {improved_forecasts['CLIENT_ID'].nunique():,}")
    print(f"  Сегменты: {sorted(improved_forecasts['SEGMENT_ID'].unique())}")
    
    if 'PRED_LOWER_BOUND' in improved_forecasts.columns:
        uncertainty_width = (improved_forecasts['PRED_UPPER_BOUND'] - 
                           improved_forecasts['PRED_LOWER_BOUND']).mean()
        print(f"  Средний uncertainty: {uncertainty_width:,.0f}")
else:
    print("Нет моделей для прогнозирования!")
    improved_forecasts = pd.DataFrame()

# %% АНАЛИЗ РЕЗУЛЬТАТОВ И CLTV
if len(improved_forecasts) > 0:
    logger.info("Анализ результатов...")
    
    cltv_by_client = improved_forecasts.groupby(['CLIENT_ID', 'SEGMENT_ID']).agg({
        'PRED_MARGIN_SURV_DISCOUNTED': 'sum',
        'PRED_MARGIN_SURV': 'sum',
        'PRED_MARGIN_RAW': 'sum'
    }).reset_index()
    
    cltv_by_client.columns = ['CLIENT_ID', 'SEGMENT_ID', 'CLTV_12M', 
                               'CLTV_12M_NO_DISCOUNT', 'CLTV_12M_RAW']
    
    if 'PRED_LOWER_BOUND' in improved_forecasts.columns:
        uncertainty_by_client = improved_forecasts.groupby('CLIENT_ID').agg({
            'PRED_LOWER_BOUND': 'sum',
            'PRED_UPPER_BOUND': 'sum'
        }).reset_index()
        
        cltv_by_client = cltv_by_client.merge(uncertainty_by_client, on='CLIENT_ID')
        cltv_by_client['CLTV_UNCERTAINTY_WIDTH'] = (
            cltv_by_client['PRED_UPPER_BOUND'] - cltv_by_client['PRED_LOWER_BOUND']
        )
    
    print("\n" + "="*70)
    print("РЕЗУЛЬТАТЫ CLTV")
    print("="*70)
    
    segment_cltv_stats = cltv_by_client.groupby('SEGMENT_ID')['CLTV_12M'].agg([
        'count', 'mean', 'median', 'std', 'min', 'max'
    ]).round(0)
    
    print("\nCLTV по сегментам:")
    print(segment_cltv_stats)
    
    print(f"\nОбщая статистика:")
    for stat, value in cltv_by_client['CLTV_12M'].describe().items():
        print(f"  {stat}: {value:,.0f}")
    
    if 'CLTV_UNCERTAINTY_WIDTH' in cltv_by_client.columns:
        print(f"\nUncertainty:")
        print(f"  Средний: {cltv_by_client['CLTV_UNCERTAINTY_WIDTH'].mean():,.0f}")
        cltv_by_client['RELATIVE_UNCERTAINTY'] = (
            cltv_by_client['CLTV_UNCERTAINTY_WIDTH'] / cltv_by_client['CLTV_12M'].abs()
        )
        print(f"  Относительный: {cltv_by_client['RELATIVE_UNCERTAINTY'].mean():.1%}")
    
    final_survival = improved_forecasts[improved_forecasts['MONTH_INDEX'] == Config.HORIZON_MONTHS]['CUM_SURVIVAL']
    print(f"\nВыживаемость после {Config.HORIZON_MONTHS}м:")
    print(f"  Средняя: {final_survival.mean():.3f}")
    print(f"  Медиана: {final_survival.median():.3f}")
    
    top_clients = cltv_by_client.nlargest(10, 'CLTV_12M')[
        ['CLIENT_ID', 'SEGMENT_ID', 'CLTV_12M']
    ]
    print(f"\nТоп-10:")
    print(top_clients.to_string(index=False))
    
    def categorize_cltv(val):
        if val < 0: return 'Убыточный'
        elif val < 100000: return 'Низкий'
        elif val < 500000: return 'Средний'
        elif val < 1000000: return 'Высокий'
        else: return 'Премиальный'
    
    cltv_by_client['CLTV_Category'] = cltv_by_client['CLTV_12M'].apply(categorize_cltv)
    
    print(f"\nРаспределение:")
    for category, count in cltv_by_client['CLTV_Category'].value_counts().items():
        pct = 100 * count / len(cltv_by_client)
        print(f"  {category}: {count:,} ({pct:.1f}%)")
    
    # 1. Детальный прогноз (все месяцы для каждого клиента)
    detailed_path = Config.MODEL_DIR / Config.MODEL_VERSION / "cltv_forecast_detailed.csv"
    improved_forecasts.to_csv(detailed_path, index=False)
    print(f"\nДетальный прогноз сохранен: {detailed_path}")
    
    # 2. Агрегированный CLTV (одна строка на клиента)
    summary_path = Config.MODEL_DIR / Config.MODEL_VERSION / "cltv_summary.csv"
    cltv_by_client.to_csv(summary_path, index=False)
    print(f"Итоговый CLTV сохранен: {summary_path}")
    
    print(f"\nМОДЕЛИРОВАНИЕ ЗАВЕРШЕНО УСПЕШНО")
    print(f"Обработано {len(cltv_by_client):,} клиентов")

# %% ЗАГРУЗКА СОХРАНЕННОЙ МОДЕЛИ (ПРИМЕР)
print("\n" + "="*70)
print("ПРИМЕР ЗАГРУЗКИ")
print("="*70)

try:
    loaded_model = ImprovedSegmentedCLTV.load_models(
        path=Config.MODEL_DIR,
        version=Config.MODEL_VERSION
    )
    
    print(f"Модель загружена")
    print(f"  Сегменты: {list(loaded_model.models.keys())}")
    
    sample_client = prod_fixed.iloc[0:1]
    sample_features = sample_client[available_features]
    
    prediction = loaded_model.predict_segment(
        segment_id=str(sample_client['SEGMENT_ID'].iloc[0]),
        X=sample_features,
        return_uncertainty=True
    )
    
    print(f"\nПример предсказания:")
    print(f"  Client: {sample_client['CLIENT_ID'].iloc[0]}")
    print(f"  Prediction: {prediction['prediction'][0]:,.0f}")
    if 'lower_bound' in prediction:
        print(f"  Bounds: [{prediction['lower_bound'][0]:,.0f}, {prediction['upper_bound'][0]:,.0f}]")

except Exception as e:
    print(f"Ошибка загрузки: {e}")

print("\nNotebook завершен!")

2025-11-10 02:13:58,394 - INFO - Запуск прогнозирования...


Данные для прогнозирования:
  Всего: 358,787
  С churn: 163,387

Прогноз на 6 месяцев...
Месяц 1/6
Ошибка 1040: Нет модели для 1040
Ошибка 1040: Нет модели для 1040
Месяц 3/6
Ошибка 1040: Нет модели для 1040
Ошибка 1040: Нет модели для 1040
Месяц 5/6
Ошибка 1040: Нет модели для 1040
Ошибка 1040: Нет модели для 1040
Завершено: 980,322 записей


2025-11-10 02:17:14,024 - INFO - Анализ результатов...



Результаты:
  Прогнозов: 980,322
  Клиентов: 163,387
  Сегменты: ['1022', '1023', '1026', '1027', '1040']

РЕЗУЛЬТАТЫ CLTV

CLTV по сегментам:
            count          mean        median            std  \
SEGMENT_ID                                                     
1022        12757  2,995,838.00    466,251.00   8,699,373.00   
1023         1032 18,577,608.00  1,939,793.00 107,406,356.00   
1026        86464    133,128.00     19,135.00     424,860.00   
1027        63133    335,082.00     74,312.00     849,309.00   
1040            1 -1,041,449.00 -1,041,449.00            NaN   

                         min            max  
SEGMENT_ID                                   
1022          -93,856,440.00 142,108,829.00  
1023       -2,392,589,112.00 783,695,013.00  
1026           -2,554,834.00   5,004,387.00  
1027           -2,125,264.00   8,849,702.00  
1040           -1,041,449.00  -1,041,449.00  

Общая статистика:
  count: 163,387
  mean: 551,173
  std: 9,039,235
  min: -2,392,58

2025-11-10 02:17:25,696 - INFO - Модели загружены: models\20251109_233743


Модель загружена
  Сегменты: ['1022', '1023', '1026', '1027']

Пример предсказания:
  Client: 18826
  Prediction: 5,671
  Bounds: [423, 75,829]

Notebook завершен!


In [5]:
train.query("TARGET_NEXT_MARGIN > 10000000")

,CLIENT_ID,MONTH_END,SEGMENT_ID,QUALITY_CODE,SUBJECT_KIND_ID,EC_SECTOR_ID,MARGIN,MARGIN_LAG1,MARGIN_LAG2,MARGIN_LAG3,MARGIN_AVG_1M_LAG,MARGIN_AVG_2M_LAG,MARGIN_AVG_3M_LAG,MARGIN_AVG_6M_LAG,MARGIN_AVG_12M_LAG,MARGIN_STDDEV_12M_LAG,MARGIN_GROWTH_RATE_3M,MONTH_OF_YEAR,QUARTER_OF_YEAR,TENURE_MONTHS,SEGMENT_AVG_MARGIN,SEGMENT_MEDIAN_MARGIN,TARGET_NEXT_MARGIN
48,1071,2025-07-31 00:00:00.000,1023,БГ,ЮЛ,1039,"5,221,297.00","3,946,911.00","9,022,725.00","2,670,606.00","3,946,911.00","6,484,818.00","5,213,414.00","3,219,689.20","1,864,515.82","2,735,508.18",5.22,7,3,31,"3,196,229.14","102,764.93","10,607,905.28"
52,1078,2024-03-31 00:00:00.000,1023,БГ,ЮЛ,1040,"8,242,035.00","6,617,804.20","26,562,957.96","22,380,438.88","6,617,804.20","16,590,381.08","18,520,400.35","15,247,603.98","12,796,577.66","6,105,812.44",0.16,3,1,15,"2,305,358.56","55,395.62","11,414,782.63"
55,1078,2024-06-30 00:00:00.000,1023,БГ,ЮЛ,1040,"7,261,600.33","9,601,015.32","11,414,782.63","8,242,035.00","9,601,015.32","10,507,898.97","9,752,610.98","12,487,719.02","12,384,984.20","6,255,586.91",-0.24,6,2,18,"2,994,878.06","47,296.02","12,048,650.40"
57,1078,2024-08-31 00:00:00.000,1023,БГ,ЮЛ,1040,"7,755,244.64","12,048,650.40","7,261,600.33","9,601,015.32","12,048,650.40","9,655,125.37","9,637,088.68","9,713,616.74","12,055,629.33","6,440,280.66",-0.01,8,3,20,"2,761,399.18","54,960.66","10,706,373.01"
58,1078,2024-09-30 00:00:00.000,1023,БГ,ЮЛ,1040,"10,706,373.01","7,755,244.64","12,048,650.40","7,261,600.33","7,755,244.64","9,901,947.52","9,021,831.79","9,616,258.66","12,051,031.66","6,443,635.15",0.05,9,3,21,"2,812,435.10","50,475.26","10,168,300.81"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4050284,11100883,2025-06-30 00:00:00.000,1023,БГ,ЮЛ,1023,"38,930,873.03","28,616,403.52","16,732,038.42","15,606,223.21","28,616,403.52","22,674,220.97","20,318,221.72","17,132,713.06","14,114,621.69","7,570,060.06",0.69,6,2,30,"2,507,416.77","93,713.07","38,867,229.49"
4050285,11100883,2025-07-31 00:00:00.000,1023,БГ,ЮЛ,1023,"38,867,229.49","38,930,873.03","28,616,403.52","16,732,038.42","38,930,873.03","33,773,638.28","28,093,104.99","22,154,236.87","16,596,246.82","10,607,692.03",1.34,7,3,31,"3,196,229.14","102,764.93","40,918,810.65"
4050286,11100883,2025-08-31 00:00:00.000,1023,БГ,ЮЛ,1023,"40,918,810.65","38,867,229.49","38,930,873.03","28,616,403.52","38,867,229.49","38,899,051.26","35,471,502.01","27,750,553.53","18,620,881.61","12,097,992.54",0.91,8,3,32,"3,442,341.94","93,182.57","44,175,578.30"
4051449,11109900,2025-07-31 00:00:00.000,1026,БГ,ЮЛ,1041,"1,779,191.40","999,670.89","1,401,390.81","4,011,006.79","999,670.89","1,200,530.85","2,137,356.17","2,811,175.46","2,517,488.49","1,679,618.88",-0.69,7,3,31,"21,305.66","1,889.25","13,022,224.16"


In [6]:
# improved_forecasts.query("CLIENT_ID in (10504212, 5360629, 1801023, 4141432, 6235067, 11142900, 10980676, 10390822, 10393200, 7454435, 7530156)")

In [10]:
from sqlalchemy import create_engine
from sqlalchemy.engine.url import URL

db_url_prod = URL.create(
    port= '6300',
    database= 'cltvdb',
    drivername= 'postgresql',
    username= 'tazhigua', 
    password= 'uJ3RqguNs0PMgt',
    host= '10.15.159.59')

_engine_prod = create_engine(db_url_prod)

In [8]:
improved_forecasts.to_sql(
        name='cltv_msb_forecast_general',
        con=_engine_prod,
        schema='public',
        if_exists='append',
        index=False
    )

322

In [9]:
query = """
select * from (
select t.*, 'eqr' as tp FROM public.cltv_msb_forecast_eqr t
union all 
select t2.*, 'loan' as tp FROM public.cltv_msb_forecast_loan t2
union all 
select t3.*, 'rko' as tp FROM public.cltv_msb_forecast_rko t3
union all 
select t4.*, 'tek_chet' as tp FROM public.cltv_msb_forecast_tek_chet  t4
union all 
select t5.*, 'usl_obz' as tp FROM public.cltv_msb_forecast_usl_obz t5) foo
where "CLIENT_ID" in (1801023,4141432,5360629,6235067,7454435,7530156,10390822,10393200,10504212,10980676,11142900)
"""

In [13]:
pd.read_sql(query,_engine_prod).to_excel("cltv_msb_november.xlsx")

In [16]:
query2 = """
select * from public.cltv_msb_forecast_general cmfg 
where "CLIENT_ID" in (1801023,4141432,5360629,6235067,7454435,7530156,10390822,10393200,10504212,10980676,11142900)
"""
pd.read_sql(query2, _engine_prod).to_excel("cltv_msb_november_main.xlsx")